# Profiling Baseline LLM Performance

This notebook establishes a baseline profile for a causal language model using `torch.profiler`.

It measures:
- End-to-end latency
- Tokens per second
- CPU and GPU memory usage
- Top profiler operators on CPU and CUDA

The notebook runs locally on CPU and automatically enables the GPU section when a CUDA device is available.

In [ ]:
import gc
import time
from dataclasses import dataclass, asdict

import psutil
import torch
import torch.profiler
from transformers import AutoModelForCausalLM, AutoTokenizer

print(f"torch: {torch.__version__}")
print(f"cuda available: {torch.cuda.is_available()}")
print(f"mps available: {getattr(torch.backends, 'mps', None) is not None and torch.backends.mps.is_available()}")

In [ ]:
model_name = "gpt2-medium"
prompt = "Alan Turing was a pioneering computer scientist whose work laid the foundation for modern computing."
num_new_tokens_to_generate = 50

tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_name)
inputs = tokenizer(prompt, return_tensors="pt")

print(f"Loaded model: {model_name}")
print(f"Input tokens: {inputs['input_ids'].shape[-1]}")
print(f"Generation tokens: {num_new_tokens_to_generate}")

In [ ]:
@dataclass
class RunMetrics:
    device: str
    wall_latency_s: float
    generated_tokens: int
    tokens_per_second: float
    model_memory_gb: float
    process_rss_gb: float
    peak_device_memory_gb: float | None = None


def run_inference(model_to_run, input_data, max_tokens, pad_id):
    with torch.no_grad():
        return model_to_run.generate(
            input_ids=input_data['input_ids'],
            attention_mask=input_data.get('attention_mask'),
            max_new_tokens=max_tokens,
            pad_token_id=pad_id,
        )


def get_model_memory_usage_gb(model_to_measure):
    model_mem_bytes = sum(p.numel() * p.element_size() for p in model_to_measure.parameters())
    return model_mem_bytes / (1024 ** 3)


def get_process_memory_usage_gb():
    gc.collect()
    return psutil.Process().memory_info().rss / (1024 ** 3)


def profile_cpu(model_to_run, input_data, max_tokens, pad_id):
    model_to_run = model_to_run.to('cpu')
    cpu_inputs = {k: v.to('cpu') for k, v in input_data.items()}

    start = time.perf_counter()
    with torch.profiler.profile(activities=[torch.profiler.ProfilerActivity.CPU]) as prof:
        outputs = run_inference(model_to_run, cpu_inputs, max_tokens, pad_id)
    end = time.perf_counter()

    generated_tokens = outputs.shape[-1] - cpu_inputs['input_ids'].shape[-1]
    metrics = RunMetrics(
        device='cpu',
        wall_latency_s=end - start,
        generated_tokens=generated_tokens,
        tokens_per_second=generated_tokens / (end - start),
        model_memory_gb=get_model_memory_usage_gb(model_to_run),
        process_rss_gb=get_process_memory_usage_gb(),
    )
    return outputs, metrics, prof


def profile_cuda(model_to_run, input_data, max_tokens, pad_id):
    if not torch.cuda.is_available():
        raise RuntimeError('CUDA is not available in this runtime.')

    model_to_run = model_to_run.to('cuda')
    gpu_inputs = {k: v.to('cuda') for k, v in input_data.items()}
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    start_event = torch.cuda.Event(enable_timing=True)
    end_event = torch.cuda.Event(enable_timing=True)

    with torch.profiler.profile(
        activities=[torch.profiler.ProfilerActivity.CPU, torch.profiler.ProfilerActivity.CUDA]
    ) as prof:
        start_event.record()
        outputs = run_inference(model_to_run, gpu_inputs, max_tokens, pad_id)
        end_event.record()
        torch.cuda.synchronize()

    latency_s = start_event.elapsed_time(end_event) / 1000.0
    generated_tokens = outputs.shape[-1] - gpu_inputs['input_ids'].shape[-1]
    metrics = RunMetrics(
        device='cuda',
        wall_latency_s=latency_s,
        generated_tokens=generated_tokens,
        tokens_per_second=generated_tokens / latency_s,
        model_memory_gb=get_model_memory_usage_gb(model_to_run),
        process_rss_gb=get_process_memory_usage_gb(),
        peak_device_memory_gb=torch.cuda.max_memory_allocated() / (1024 ** 3),
    )
    return outputs, metrics, prof


def show_profiler_table(prof, sort_by, row_limit=15):
    print(prof.key_averages().table(sort_by=sort_by, row_limit=row_limit))

In [ ]:
cpu_outputs, cpu_metrics, prof_cpu = profile_cpu(
    model,
    inputs,
    num_new_tokens_to_generate,
    tokenizer.pad_token_id,
)

print('CPU metrics:')
for key, value in asdict(cpu_metrics).items():
    print(f"  {key}: {value}")

print('\nGenerated text (CPU):')
print(tokenizer.decode(cpu_outputs[0], skip_special_tokens=True))

In [ ]:
show_profiler_table(prof_cpu, sort_by='cpu_time_total')

In [ ]:
gpu_metrics = None
prof_gpu = None

if torch.cuda.is_available():
    gpu_outputs, gpu_metrics, prof_gpu = profile_cuda(
        model,
        inputs,
        num_new_tokens_to_generate,
        tokenizer.pad_token_id,
    )
    print('GPU metrics:')
    for key, value in asdict(gpu_metrics).items():
        print(f"  {key}: {value}")

    print('\nGenerated text (GPU):')
    print(tokenizer.decode(gpu_outputs[0], skip_special_tokens=True))
else:
    print('CUDA is not available in this runtime. GPU profiling is skipped.')
    print('On Apple Silicon, you can still run CPU profiling locally.')

In [ ]:
if prof_gpu is not None:
    show_profiler_table(prof_gpu, sort_by='cuda_time_total')

In [ ]:
print('Summary')
print('-------')
print(f"CPU latency: {cpu_metrics.wall_latency_s:.4f} s")
print(f"CPU throughput: {cpu_metrics.tokens_per_second:.2f} tokens/s")

if gpu_metrics is not None:
    print(f"GPU latency: {gpu_metrics.wall_latency_s:.4f} s")
    print(f"GPU throughput: {gpu_metrics.tokens_per_second:.2f} tokens/s")
    print(f"GPU peak memory: {gpu_metrics.peak_device_memory_gb:.4f} GB")
else:
    print('GPU metrics unavailable in this runtime.')